In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

GEMINI_MODEL= "gemini-3.1-flash-lite"

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
llm = GoogleGenerativeAI(model=GEMINI_MODEL)

/tmp/ipykernel_14271/1435967175.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

In [6]:
# Load documents from the URLs
docs = WebBaseLoader(urls).load()

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs)
print(f"Number of document chunks created: {len(doc_splits)}")

# Add the document chunks to the vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings,
)

print("Data loaded and vector store created successfully.")

# Create a retriever component
retriever = vectorstore.as_retriever(k=6)

Number of document chunks created: 88
Data loaded and vector store created successfully.


In [7]:
from langsmith import traceable

@traceable()
def rag_bot(question: str) -> dict:
    # Retrieve relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at \
analyzing source information and answering questions. \
Use the following source documents to answer the user's questions. \
If you don't know the answer, just say that you don't know. \
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    # Generate answer
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])
    return {"answer": ai_msg, "documents": docs}

In [8]:
rag_bot("what is agents")

{'answer': "In an LLM-powered autonomous agent system, the large language model functions as the agent's brain to act as a general problem solver. These agents utilize planning, memory, and tool use to break down tasks, retain information, and access external data or capabilities. They can operate independently to achieve specific goals, as demonstrated by proof-of-concept examples like AutoGPT.",
 'documents': [Document(id='229f1715-7735-4f73-ad15-dfec280ed42e', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powe

## Setting Evaluation

### Creating dataset

In [1]:
from langsmith import Client

client = Client()

examples = [
    {
        "inputs": {
            "question": "How does the ReAct agent use self-reflection?"
        },
        "outputs": {
            "answer": "ReAct integrates reasoning and acting, performing "
            "actions - such tools like Wikipedia search API - and then "
            "observing / reasoning about the tool outputs."
        },
    },
    {
        "inputs": {
            "question": "What are the types of biases that can arise "
            "with few-shot prompting?"
        },
        "outputs": {
            "answer": "The biases that can arise with few-shot prompting "
            "include (1) Majority label bias, (2) Recency bias, and "
            "(3) Common token bias."
        },
    },
    {
        "inputs": {
            "question": "What are five types of adversarial attacks?"
        },
        "outputs": {
            "answer": "Five types of adversarial attacks are "
            "(1) Token manipulation, (2) Gradient based attack, "
            "(3) Jailbreak prompting, (4) Human red-teaming, "
            "(5) Model red-teaming."
        },
    },
]

dataset_name = "RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(dataset_id=dataset.id, examples=examples)

{'example_ids': ['7162b7a0-ca54-4ac1-a157-e6cc77459a21',
  '7e37e19a-221e-4c58-8893-f9f78c527c90',
  'bf0ddd88-c862-4ab9-bd46-54886031469b'],
 'count': 3,
 'as_of': '2026-06-02T09:40:03.34043247Z'}

### Initializing evalution metrics

#### Correctness (Response vs Reference Answer)

In [11]:
from typing import Annotated, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI

class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, "Explanation of the grading decision."]
    correct: Annotated[bool, "True if the answer is correct, False otherwise."]

correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, \
and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy \
relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting \
statements.
(3) It is OK if the student answer contains more information than \
the ground truth answer, as long as it is factually accurate \
relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets \
all of the criteria.
A correctness value of False means that the student's answer does \
not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your \
reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

grader_llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0).with_structured_output(CorrectnessGrade,  method="json_schema", strict=True)

def correctness(
    inputs: dict, outputs: dict, reference_outputs: dict
) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers},
    ])
    return grade["correct"]

#### Relevance - Response vs input

In [12]:
class RelevanceGrade(TypedDict):
    explanation: Annotated[
        str, ..., "Explain your reasoning for the score"
    ]
    relevant: Annotated[
        bool, ...,
        "Provide the score on whether the answer addresses the question",
    ]

relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets \
all of the criteria.
A relevance value of False means that the student's answer does \
not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your \
reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

relevance_llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL, temperature=0
).with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)

def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = (
        f"QUESTION: {inputs['question']}\n"
        f"STUDENT ANSWER: {outputs['answer']}"
    )
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["relevant"]

#### grounded - response vs retrieved documents

In [13]:
class GroundedGrade(TypedDict):
    explanation: Annotated[
        str, ..., "Explain your reasoning for the score"
    ]
    grounded: Annotated[
        bool, ...,
        "Provide the score on if the answer hallucinates from the documents",
    ]

grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" \
information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets \
all of the criteria.
A grounded value of False means that the student's answer does \
not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your \
reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

grounded_llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL, temperature=0
).with_structured_output(
    GroundedGrade, method="json_schema", strict=True
)

def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(
        doc.page_content for doc in outputs["documents"]
    )
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["grounded"]

#### Retrieval Relevance (Retrieved Docs vs Input)

In [16]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[
        str, ..., "Explain your reasoning for the score"
    ]
    relevant: Annotated[
        bool, ...,
        "True if the retrieved documents are relevant to the question, "
        "False otherwise",
    ]

retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated \
to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related \
to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated \
to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords \
or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely \
unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your \
reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

retrieval_relevance_llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL, temperature=0
).with_structured_output(
    RetrievalRelevanceGrade, method="json_schema", strict=True
)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(
        doc.page_content for doc in outputs["documents"]
    )
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["relevant"]

### Executing Evaluation Pipeline

In [18]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[
        correctness,
        groundedness,
        relevance,
        retrieval_relevance,
    ],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gemini 3.5 flash"},
)

# Explore results locally as a dataframe
res_df = experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-3be4e2cd' at:
https://smith.langchain.com/o/9ca388df-7822-4c9c-bfe3-6c188c01735f/datasets/c9a0d20f-aa10-45ba-bc95-8681f854574b/compare?selectedSessions=47c22e2f-8fca-4ebf-b597-66a9cbb73b03




3it [00:19,  6.54s/it]


In [20]:
res_df.to_csv("experiment_results.csv", index=False)

In [29]:
print(res_df[['inputs.question','outputs.documents']]['outputs.documents'][0][0].page_content)

Examples of reasoning trajectories for knowledge-intensive tasks (e.g. HotpotQA, FEVER) and decision-making tasks (e.g. AlfWorld Env, WebShop). (Image source: Yao et al. 2023).

In both experiments on knowledge-intensive tasks and decision-making tasks, ReAct works better than the Act-only baseline where Thought: … step is removed.
Reflexion (Shinn & Labash 2023) is a framework to equip agents with dynamic memory and self-reflection capabilities to improve reasoning skills. Reflexion has a standard RL setup, in which the reward model provides a simple binary reward and the action space follows the setup in ReAct where the task-specific action space is augmented with language to enable complex reasoning steps. After each action $a_t$, the agent computes a heuristic $h_t$ and optionally may decide to reset the environment to start a new trial depending on the self-reflection results.


Illustration of the Reflexion framework. (Image source: Shinn & Labash, 2023)

The heuristic function d